# Agentic RAG: 에이전트 기반 RAG - 실습 코드 1: Agentic RAG 구현 (상세 설명 버전)

- Tutorial ID: `expand-agentic-rag`
- Tutorial: Agentic RAG: 에이전트 기반 RAG
- Section ID: `expand-agentic-rag-code-1`
- Section: 실습 코드 1: Agentic RAG 구현 (간략화)

> 이 노트북은 원본 실습 코드에 있던 오류(잘못된 들여쓰기, 정의되지 않은 함수 호출 등)를 모두 고치고,
> **실제로 실행 가능한 예제**와 **각 단계별 상세한 설명·주석**을 추가한 버전입니다.
> RAG나 에이전트 개념을 처음 접하는 분도 위에서부터 순서대로 실행하며 따라올 수 있도록 구성했습니다.


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: Agentic RAG 구현 (상세 설명 버전)
#
# 이 노트북은 "정답 코드를 한 번 실행"하는 용도가 아니라,
# Agentic RAG의 핵심 아이디어인
#   "검색 여부와 검색 방법을 LLM이 스스로 판단한다"
# 는 개념이 실제 코드로 어떻게 구현되는지 한 단계씩 추적하기 위한 실습 노트입니다.
#
# 학습 목표:
#   1) 질문 -> [검색 필요 여부 판단] -> [검색 전략 선택] -> [검색 실행] -> [답변 생성]
#      으로 이어지는 파이프라인의 각 단계를 코드로 직접 확인합니다.
#   2) 실제 LLM/벡터DB API 없이도 동작 원리를 이해할 수 있도록,
#      Mock(가짜) LLM과 Mock 벡터스토어를 직접 만들어 사용합니다.
#   3) 같은 "질문을 던진다"는 행동이라도, 질문의 생김새에 따라
#      서로 다른 전략이 선택되는 과정을 직접 관찰합니다.
#
# 읽는 순서:
#   1) 먼저 MockLLM과 MockVectorStore를 단독으로 실행해보며 "왜 필요한지" 감을 잡습니다.
#   2) SearchStrategy(검색 전략 선택지)가 무엇을 의미하는지 확인합니다.
#   3) AgenticRAG 클래스의 각 메서드(should_retrieve, select_strategy, retrieve, query)를
#      하나씩 따로 호출해보며 입력과 출력을 확인합니다.
#   4) 마지막으로 전체 파이프라인(query 메서드)을 4가지 예시 질문에 실행하며,
#      질문 유형에 따라 파이프라인이 어떻게 다르게 동작하는지 비교합니다.
#   5) 예시 질문·문서·키워드를 직접 바꿔보며 결과가 어떻게 달라지는지 실험해봅니다.
#
# 주의:
#   - 이 노트북의 MockLLM/MockVectorStore는 "개념 이해"를 위한 간단한 규칙 기반 구현입니다.
#     실제 서비스에서는 OpenAI/Claude 같은 LLM API와 Chroma/Pinecone 같은 벡터 DB를 사용합니다.
#     (맨 마지막 "정리 및 다음 단계"에서 실제 API로 바꾸는 방법을 간단히 소개합니다.)
#   - 코드 중간의 print 출력(verbose 로그)을 꼭 함께 읽으며 "지금 어느 단계인지" 확인하세요.
#   - 셀은 반드시 위에서부터 순서대로 실행하세요. (뒤 셀은 앞 셀에서 정의한 클래스/변수를 사용합니다)
# ============================================================


## 0. Agentic RAG란 무엇인가요?

### 0-1. 기존 RAG의 한계

**RAG(Retrieval-Augmented Generation, 검색 증강 생성)**는 LLM이 답변을 만들기 전에
관련 문서를 검색해서 그 내용을 참고하게 만드는 기법입니다.

그런데 **기존(Naive) RAG**는 질문의 내용과 상관없이 아래 순서를 기계적으로 반복합니다.

```
질문 입력 -> (무조건) 문서 검색 -> 검색된 문서로 답변 생성
```

문제는, 세상의 모든 질문이 검색을 필요로 하지는 않는다는 점입니다.

- "안녕?" 같은 인사말에도 매번 문서를 검색하러 가야 할까요?
- "휴가 정책이 뭐야?"처럼 사실 하나를 묻는 질문과, "휴가는 어떻게 신청하고 신청 후 며칠 안에
  승인이 나?"처럼 여러 하위 질문이 섞인 질문을 **똑같은 방법**으로 검색해도 괜찮을까요?

### 0-2. Agentic RAG의 아이디어: "판단하는 사서"

**비유**: 도서관에 두 명의 사서가 있다고 생각해봅시다.

- **기계적인 사서 (기존 RAG)**: 무슨 질문을 받든 일단 서고로 달려가 책을 찾아온 뒤,
  그 책만 보고 대답합니다. "지금 몇 시예요?"라고 물어도 서고로 달려갑니다.
- **똑똑한 사서 (Agentic RAG)**: 질문을 먼저 듣고 판단합니다.
  - "이건 내가 바로 답할 수 있는 질문이네" -> 바로 대답
  - "이건 찾아봐야 하는데, 어떤 방식으로 찾는 게 좋을까?" -> 상황에 맞는 검색 방법을 고른 뒤 검색

**Agentic RAG**는 이 "똑똑한 사서"의 판단 과정을 LLM에게 맡깁니다.
즉, 검색 여부와 검색 방법 자체를 LLM이 스스로 결정하는 "에이전트(agent)"처럼 행동하게 만드는 것입니다.

### 0-3. 한눈에 비교

| 구분 | 기존(Naive) RAG | Agentic RAG |
|---|---|---|
| 검색 여부 | 항상 검색함 | 필요할 때만 검색함 |
| 검색 방법 | 고정된 한 가지 방법 | 질문에 따라 방법을 선택함 |
| 복합 질문 처리 | 질문 그대로 1회 검색 | 하위 질문으로 쪼개서 여러 번 검색 가능 |
| 유연성 | 낮음 | 높음 (상황에 맞게 판단) |

### 0-4. 이 노트북이 구현할 파이프라인

```
질문 입력
   |
   v
[1단계] should_retrieve(query)  ->  "이 질문에 답하려면 문서 검색이 필요한가?"
   |
   +-- No  ------------------------------->  검색 없이 LLM이 바로 답변 (종료)
   |
   +-- Yes
       |
       v
   [2단계] select_strategy(query)  ->  "어떤 방식으로 검색할까?"
       |
       +-- SEMANTIC     : 질문 그대로 1회, 의미 기반 검색
       +-- MULTI_QUERY  : 질문을 하위 질문 여러 개로 쪼개어 각각 검색
       +-- HYBRID       : 의미 기반 검색 + 키워드 검색을 함께 사용
       |
       v
   [3단계] retrieve(query, strategy)  ->  선택된 전략으로 실제 문서 검색
       |
       v
   [4단계] context(검색 결과) + question(질문) -> 프롬프트 구성 -> LLM 답변 생성
       |
       v
      최종 답변
```

이제 위 그림의 각 상자를 실제 코드로 하나씩 만들어보겠습니다.


## 1. 왜 가짜(Mock) LLM과 벡터스토어를 사용하나요?

실전에서 Agentic RAG를 만들려면 보통 아래 두 가지가 필요합니다.

1. **LLM API** — 질문에 대한 판단/답변을 생성해주는 모델 (예: Claude, GPT 등)
2. **벡터스토어(Vector Store)** — 문서를 임베딩(embedding) 벡터로 저장해두고,
   질문과 의미가 비슷한 문서를 찾아주는 저장소 (예: Chroma, Pinecone, FAISS 등)

하지만 이 둘은 보통 **API 키, 네트워크 연결, 별도 설치**가 필요해서,
"Agentic RAG의 핵심 로직(파이프라인 구조)을 이해"하는 데는 오히려 방해가 될 수 있습니다.

그래서 이 노트북에서는 실제 LLM과 벡터스토어의 **역할만 흉내 내는 간단한 가짜(Mock) 클래스**를
직접 만들어 사용합니다. 이렇게 하면:

- API 키 없이, 인터넷 연결 없이 바로 실행하고 결과를 볼 수 있습니다.
- 내부 동작이 100% 투명하기 때문에(우리가 직접 규칙을 정했으므로), "왜 이런 결과가 나왔는지" 완벽하게 추적할 수 있습니다.
- 나중에 이 Mock 클래스를 실제 API 호출 코드로 바꾸기만 하면, `AgenticRAG` 클래스 자체는 전혀 수정할 필요가 없습니다.
  (이런 식으로 설계하는 이유는 맨 마지막 "정리 및 다음 단계"에서 자세히 다룹니다.)


In [ ]:
# List, Dict : 타입을 명시하기 위한 타입 힌트(type hint)용 도구입니다.
#   예) def f(x: int) -> str 처럼 "이 함수는 int를 받아서 str을 반환한다"고 표시해두면,
#       코드를 읽는 사람이 입력/출력 형태를 바로 알 수 있습니다. (실행에는 영향을 주지 않습니다)
# Enum : "정해진 선택지 중 하나만 가질 수 있는 값"을 만들 때 사용합니다. (4장에서 자세히 다룹니다)
# json : 아래에서 검색 결과(딕셔너리 리스트)를 보기 좋게 출력할 때 사용합니다.
from enum import Enum
from typing import List, Dict
import json

print("준비 완료: Enum, List, Dict, json을 불러왔습니다.")


## 2. MockLLM — 가짜 LLM 만들기

실제 LLM은 어떤 프롬프트(지시문)를 주더라도 문맥을 이해해서 자연스러운 답을 만들어냅니다.
`MockLLM`은 이걸 완벽히 흉내 낼 수는 없지만, 대신 다음과 같은 **간단한 규칙**으로 동작합니다.

> 프롬프트 안에 특정 문구가 들어있는지 확인하고, 그 문구에 따라 다른 방식으로 응답한다.

예를 들어 프롬프트 안에 `"require external knowledge"`라는 문구가 있으면
"이건 검색 필요 여부를 묻는 질문이구나"라고 판단하고 그에 맞는 답(`yes`/`no`)을 돌려줍니다.

> **여기서 배우는 것**: 같은 LLM이라도 프롬프트(지시문)를 어떻게 작성하느냐에 따라
> 완전히 다른 역할(판단자, 검색 전략가, 답변자 등)을 수행하게 만들 수 있습니다.
> 이것이 바로 **프롬프트 엔지니어링**의 핵심 아이디어이며, `AgenticRAG` 클래스가
> 하나의 `llm` 객체만으로 여러 단계의 의사결정을 처리할 수 있는 이유입니다.

아래 코드에서 각 메서드는 서로 다른 "프롬프트 유형"을 처리합니다. 표로 정리하면:

| 프롬프트에 포함된 문구 | 처리하는 메서드 | 역할 |
|---|---|---|
| `"require external knowledge"` | `_decide_retrieval` | 검색 필요 여부(yes/no) 판단 |
| `"Select search strategy"` | `_decide_strategy` | 검색 전략(semantic/multi_query/hybrid) 선택 |
| `"Break down this question"` | `_decompose_query` | 복합 질문을 하위 질문으로 분리 |
| `"Based on context, answer the question"` | `_answer_with_context` | 검색된 문서를 참고해 최종 답변 생성 |
| (위 어디에도 해당 없음) | `_chit_chat` | 검색이 필요 없는 일반 대화 응답 |


In [ ]:
class MockLLM:
    """
    실제 LLM(Claude, GPT 등)을 흉내 내는 가짜 모델입니다.

    실제 프로젝트라면 이 클래스 대신 아래와 비슷한 코드를 사용하게 됩니다.
        import anthropic
        client = anthropic.Anthropic()
        client.messages.create(model="...", messages=[...])

    이 노트북에서는 API 키 없이 파이프라인 구조 자체에 집중하기 위해,
    프롬프트 안의 '특정 문구'를 확인해서 미리 정해둔 규칙대로 응답하는
    간단한 가짜 버전을 사용합니다.
    """

    def __init__(self, verbose: bool = True):
        # verbose=True로 설정하면, LLM이 어떤 프롬프트를 받고 어떤 응답을 돌려주는지
        # 매번 화면에 출력해줍니다. (파이프라인 내부를 들여다보기 위한 옵션)
        self.verbose = verbose

    def __call__(self, prompt: str) -> str:
        """
        객체를 함수처럼 호출할 수 있게 해주는 매직 메서드입니다.
        즉, llm = MockLLM()을 만든 뒤 llm("질문")처럼 바로 괄호를 붙여 호출할 수 있습니다.
        (뒤에서 만들 AgenticRAG 코드가 self.llm(prompt)라고 호출하는 부분이 바로 이 메서드를 실행시킵니다.)
        """
        if self.verbose:
            print("\n[LLM 호출] 아래와 같은 프롬프트가 전달되었습니다:")
            print("-" * 50)
            print(prompt)
            print("-" * 50)

        response = self._generate(prompt)

        if self.verbose:
            print(f"[LLM 응답] {response}")

        return response

    def _generate(self, prompt: str) -> str:
        # 프롬프트 안에 어떤 문구가 들어있는지에 따라 처리 방식을 나눕니다.
        # (실제 LLM은 이런 문자열 매칭 없이 문맥을 통째로 이해하지만,
        #  여기서는 학습 목적상 프롬프트 템플릿별로 분기 처리합니다.)
        if "require external knowledge" in prompt:
            return self._decide_retrieval(prompt)
        elif "Select search strategy" in prompt:
            return self._decide_strategy(prompt)
        elif "Break down this question" in prompt:
            return self._decompose_query(prompt)
        elif "Based on context, answer the question" in prompt:
            return self._answer_with_context(prompt)
        else:
            # 위 네 가지 형식에 해당하지 않으면, 검색이 필요 없는 일반 대화로 간주합니다.
            return self._chit_chat(prompt)

    def _decide_retrieval(self, prompt: str) -> str:
        """'검색이 필요한가?'를 판단합니다. (should_retrieve에서 사용)"""
        # 프롬프트는 f"...'{query}' Answer yes/no." 형태이므로,
        # 작은따옴표(') 사이에 있는 문자열이 바로 원본 질문입니다.
        query = prompt.split("'")[1]

        # 인사말/잡담에 흔히 등장하는 단어가 있으면 검색이 필요 없다고 판단합니다.
        chit_chat_keywords = ["안녕", "반가워", "고마워", "잘 지내", "기분", "너는 누구"]
        if any(keyword in query for keyword in chit_chat_keywords):
            return "no"
        return "yes"

    def _decide_strategy(self, prompt: str) -> str:
        """검색 전략(semantic/multi_query/hybrid)을 선택합니다. (select_strategy에서 사용)"""
        query = prompt.split("'")[1]

        # '그리고', '또한'처럼 여러 하위 질문을 이어붙이는 연결어가 있으면 -> 복합 질문
        multi_signals = ["그리고", "또한", "각각"]
        # '번호', '내선'처럼 정확한 값(고유 정보)을 콕 집어 묻는 단어가 있으면 -> 정확한 키워드 매칭 필요
        exact_signals = ["번호", "몇 시", "내선"]

        has_multi = any(signal in query for signal in multi_signals)
        has_exact = any(signal in query for signal in exact_signals)

        if has_multi:
            return "multi_query"
        elif has_exact:
            return "hybrid"
        else:
            # 위 두 신호가 모두 없으면, 하나의 사실을 묻는 단순한 질문으로 판단합니다.
            return "semantic"

    def _decompose_query(self, prompt: str) -> str:
        """복합 질문을 여러 개의 하위 질문으로 쪼갭니다. (multi_query 전략에서 사용)"""
        query = prompt.split("'")[1]

        # '그리고'/'또한'을 하나의 구분자로 통일한 뒤, 그 구분자를 기준으로 문장을 나눕니다.
        for connector in ["그리고", "또한"]:
            query = query.replace(connector, "@@SPLIT@@")

        sub_questions = [q.strip() for q in query.split("@@SPLIT@@") if q.strip()]
        return "\n".join(sub_questions)

    def _answer_with_context(self, prompt: str) -> str:
        """검색된 문서(context)를 참고해서 최종 답변을 생성합니다."""
        # 프롬프트 형식: "...Context: {context}\nQuestion: {question}\nAnswer:"
        # 이므로 "Context: "와 "Question: " 사이 문자열이 검색된 문서(context)입니다.
        context_part = prompt.split("Context: ")[1].split("Question: ")[0].strip()
        question_part = prompt.split("Question: ")[1].split("Answer:")[0].strip()

        return (f"검색된 문서에 따르면: {context_part} "
                f"\n-> 즉, '{question_part}'에 대한 답은 위 내용을 참고하시면 됩니다.")

    def _chit_chat(self, prompt: str) -> str:
        """검색 없이 바로 답할 수 있는 일반적인 대화에 응답합니다."""
        if "기분" in prompt or "안녕" in prompt:
            return "안녕하세요! 저는 오늘도 활기차게 작동 중입니다. 무엇을 도와드릴까요?"
        return "네, 알겠습니다. 별도의 문서 검색 없이 답변드릴게요."


print("MockLLM 클래스 정의 완료.")


### 2-1. MockLLM 단독으로 호출해보기

`AgenticRAG` 안에 넣기 전에, `MockLLM`이 프롬프트에 따라 어떻게 다르게 응답하는지 먼저 확인해봅시다.


In [ ]:
# 새로운 MockLLM 인스턴스를 만듭니다. 지금은 verbose를 꺼서(False) 결과만 깔끔하게 봅니다.
llm = MockLLM(verbose=False)

# 테스트 1: 그냥 평범한 질문처럼 호출 (특별한 형식의 프롬프트가 아님)
print("[테스트 1] 평범한 대화:")
print(" ->", llm("이 프로젝트 정말 재미있는 것 같아!"))

print()

# 테스트 2: '검색 필요 여부'를 판단하는 프롬프트 형식으로 호출
# (뒤에서 만들 AgenticRAG.should_retrieve 메서드 내부에서 실제로 이런 프롬프트를 만듭니다)
print("[테스트 2] 검색 필요 여부 판단용 프롬프트:")
print(" ->", llm("Does this question require external knowledge? '연차는 며칠인가요?' Answer yes/no."))

print()
print("=> 같은 MockLLM 객체라도, 프롬프트의 '형식'이 다르면 완전히 다른 역할을 수행하는 것을 확인했습니다.")


## 3. MockVectorStore — 가짜 벡터스토어 만들기

실제 벡터스토어는 다음과 같이 동작합니다.

1. 문서를 문장 임베딩 모델에 통과시켜 **벡터(숫자 배열)**로 변환해 저장해둡니다.
2. 질문이 들어오면, 질문도 같은 방식으로 벡터로 변환합니다.
3. 저장된 문서 벡터들과 질문 벡터 사이의 **코사인 유사도**를 계산해서,
   가장 가까운(=의미가 비슷한) 문서 순으로 결과를 반환합니다.

이 방식의 장점은, "휴가"와 "연차"처럼 **글자는 다르지만 의미가 비슷한 단어**도
실제 임베딩에서는 벡터 공간에서 가까운 위치에 놓이기 때문에 서로 연결해서 찾아낼 수 있다는 점입니다.

### 이 노트북의 Mock 버전이 갖는 한계 (꼭 읽어주세요)

이 노트북의 `MockVectorStore`는 임베딩 모델 없이 개념만 익히기 위해,
각 문서에 미리 정해둔 `keywords`(마치 "이 문서를 대표하는 태그") 중
몇 개가 질문 문장에 등장하는지를 세어 "유사도 점수"처럼 사용합니다.

**즉, 이 Mock은 진짜 의미 기반 검색이 아니라 "정해진 키워드가 등장하는지"를 세는 수준의 흉내입니다.**
실제 임베딩 기반 semantic search는 단어가 하나도 겹치지 않아도 의미가 비슷하면 찾아낼 수 있다는 점에서
이 예제보다 훨씬 강력합니다. 여기서는 파이프라인의 "구조"에 집중하기 위해 단순화했다는 점을 기억해주세요.

### 예제 문서 (지식 베이스)

아래처럼 5개의 사내 규정 문서를 미리 준비해두었습니다. (실전이라면 회사 위키나 매뉴얼 PDF 등에서 가져오겠죠?)

| 문서 | 주요 키워드 |
|---|---|
| 연차 휴가 정책 | 연차, 휴가, 며칠, 일수, 이월 |
| 리모트 근무 정책 | 재택근무, 리모트, 원격, 신청 |
| 출장 경비 규정 | 출장, 경비, 숙박비, 식비, 한도 |
| 비밀번호 초기화 방법 | 비밀번호, 초기화, IT, 헬프데스크, 내선번호 |
| 회의실 예약 방법 | 회의실, 예약, 미팅룸 |


In [ ]:
class MockVectorStore:
    """
    실제 벡터스토어(Chroma, Pinecone, FAISS 등)를 흉내 낸 가짜 저장소입니다.

    문서 하나하나를 {"content": 문서 본문, "keywords": 이 문서를 대표하는 단어들}
    형태로 저장해둡니다. keywords는 "실제 임베딩 모델이 학습했을 법한 의미 연결"을
    사람이 미리 정해둔 것이라고 생각하면 됩니다.
    """

    def __init__(self):
        self.documents = [
            {
                "content": "연차 휴가 정책: 입사 1년 미만 직원은 매월 1일의 연차가 발생하며, "
                           "1년 이상 근무한 직원은 매년 15일의 연차가 부여됩니다. "
                           "연차는 최대 3년까지 이월 가능합니다.",
                "keywords": ["연차", "휴가", "며칠", "일수", "이월"],
            },
            {
                "content": "리모트 근무 정책: 직원은 주 3일까지 재택근무를 신청할 수 있습니다. "
                           "재택근무 신청은 최소 3일 전에 팀장에게 승인받아야 합니다.",
                "keywords": ["재택근무", "리모트", "원격", "신청"],
            },
            {
                "content": "출장 경비 규정: 국내 출장 시 1일 숙박비 상한액은 15만원이며, "
                           "식비는 1일 5만원까지 지원됩니다. 해외 출장은 별도 규정을 따릅니다.",
                "keywords": ["출장", "경비", "숙박비", "식비", "한도"],
            },
            {
                "content": "비밀번호 초기화 방법: 사내 시스템 비밀번호를 잊어버린 경우, "
                           "IT 헬프데스크(내선 1234)로 문의하거나 사내 포털의 "
                           "'비밀번호 찾기' 메뉴를 이용하세요.",
                "keywords": ["비밀번호", "초기화", "IT", "헬프데스크", "내선번호", "내선"],
            },
            {
                "content": "회의실 예약 방법: 회의실은 사내 예약 시스템을 통해 최대 2주 전부터 "
                           "예약 가능하며, 30분 단위로 예약할 수 있습니다.",
                "keywords": ["회의실", "예약", "미팅룸"],
            },
        ]

    def similarity_search(self, query: str, k: int = 3) -> List[Dict]:
        """
        '의미 기반(semantic)' 검색을 흉내 냅니다.
        각 문서의 keywords 중 질문에 등장하는 개수를 세어 점수로 사용하고,
        점수가 높은 순으로 최대 k개 문서를 반환합니다.
        """
        scored = []
        for doc in self.documents:
            score = sum(1 for keyword in doc["keywords"] if keyword in query)
            scored.append((score, doc))

        # 점수(score) 기준 내림차순 정렬
        scored.sort(key=lambda pair: pair[0], reverse=True)

        # 점수가 0보다 큰(=조금이라도 관련 있는) 문서만, 최대 k개까지만 반환
        results = [
            {"content": doc["content"], "score": score}
            for score, doc in scored if score > 0
        ][:k]
        return results

    def keyword_search(self, query: str, k: int = 3) -> List[Dict]:
        """
        '키워드 기반(lexical)' 검색을 흉내 냅니다.
        similarity_search와 달리 미리 정의된 keywords가 아니라,
        질문의 단어가 문서 '원문'에 그대로(부분 일치) 포함되는지를 확인합니다.
        (실제 서비스에서는 BM25 같은 알고리즘이 널리 쓰입니다.)
        """
        results = []
        # 질문을 공백 기준으로 나눈 뒤, 의미 없는 짧은 글자(조사 등)를 줄이기 위해 2글자 이상만 사용
        query_tokens = [token for token in query.replace("?", "").split() if len(token) >= 2]

        for doc in self.documents:
            matched = sum(1 for token in query_tokens if token in doc["content"])
            if matched > 0:
                results.append({"content": doc["content"], "score": matched})

        results.sort(key=lambda d: d["score"], reverse=True)
        return results[:k]


print("MockVectorStore 클래스 정의 완료.")


### 3-1. MockVectorStore 단독으로 호출해보기

`similarity_search`와 `keyword_search`가 실제로 어떤 결과(어떤 모양의 데이터)를 반환하는지 직접 확인해봅시다.
반환값은 `[{"content": ..., "score": ...}, ...]` 형태의 **딕셔너리 리스트**입니다.


In [ ]:
vectorstore = MockVectorStore()

# json.dumps에 ensure_ascii=False를 꼭 붙여주세요!
# 그렇지 않으면 한글이 "\uc5f0\ucc28" 처럼 유니코드 이스케이프 코드로 출력됩니다.
print("[similarity_search 결과] '연차는 며칠 주어지나요?'")
results = vectorstore.similarity_search("연차는 며칠 주어지나요?", k=2)
print(json.dumps(results, ensure_ascii=False, indent=2))

print()
print("[keyword_search 결과] '비밀번호 내선번호'")
results2 = vectorstore.keyword_search("비밀번호 내선번호", k=2)
print(json.dumps(results2, ensure_ascii=False, indent=2))


## 4. SearchStrategy — 검색 전략을 표현하는 Enum

### 4-1. Enum(열거형)이 뭔가요?

`Enum`은 "정해진 선택지 중 하나만 값으로 가질 수 있게" 해주는 파이썬 도구입니다.
문자열을 직접 쓰는 대신 Enum을 사용하는 이유를 예로 살펴봅시다.

```python
# Enum 없이 문자열만 사용한다면...
strategy = "semantic"   # 정상
strategy = "sematic"    # 오타! 그런데 파이썬은 에러를 내지 않고 조용히 잘못된 문자열을 저장합니다.
                         # 나중에 strategy == "semantic" 비교가 계속 False로 나와야 그제서야 원인을 찾게 됩니다.
```

```python
# Enum을 사용하면...
class SearchStrategy(Enum):
    SEMANTIC = "semantic"
    MULTI_QUERY = "multi_query"
    HYBRID = "hybrid"

strategy = SearchStrategy.SEMANTIC     # 정상, 코드 자동완성도 지원됩니다.
# strategy = SearchStrategy.SEMANTC    # 오타를 내면 -> AttributeError가 즉시 발생해서 바로 알아챌 수 있습니다.
```

즉, Enum은 "이 값은 반드시 정해진 후보들(semantic/multi_query/hybrid) 중 하나여야 한다"는
규칙을 코드 차원에서 강제해줍니다. 오타를 실행 즉시 잡아낼 수 있어 훨씬 안전합니다.

### 4-2. 이 노트북에서 사용할 세 가지 전략

| 전략 | 값 | 언제 사용하나요? |
|---|---|---|
| `SEMANTIC` | `"semantic"` | 하나의 명확한 사실을 묻는 단순한 질문 |
| `MULTI_QUERY` | `"multi_query"` | 여러 개의 하위 질문이 섞인 복합 질문 |
| `HYBRID` | `"hybrid"` | 개념 설명 + 정확한 숫자/코드 등을 동시에 요구하는 질문 |

원본 코드에는 `MULTI_QUERY` 줄의 들여쓰기가 잘못되어 있어 `IndentationError`가 발생했습니다.
아래 코드에서는 세 값이 모두 같은 들여쓰기로 정리되어 있습니다.


In [ ]:
class SearchStrategy(Enum):
    SEMANTIC = "semantic"          # 의미 기반 검색: 질문을 그대로 1회 검색
    MULTI_QUERY = "multi_query"    # 복합 질문 분해: 하위 질문 여러 개로 쪼개어 각각 검색
    HYBRID = "hybrid"              # 혼합 검색: 의미 기반 + 키워드 기반을 함께 사용


print("SearchStrategy 정의 완료. 사용 가능한 값들:")
for s in SearchStrategy:
    print(f"  - {s.name} = \"{s.value}\"")


In [ ]:
# Enum의 오타 방지 효과를 직접 확인해봅시다.
try:
    wrong = SearchStrategy("sematic")  # 일부러 오타를 냄
except ValueError as e:
    print(f"[예상된 에러 발생] {e}")
    print("=> 이렇게 즉시 에러가 발생하기 때문에, 오타를 놓치지 않고 바로 고칠 수 있습니다.")

print()
correct = SearchStrategy("semantic")
print(f"정상 값: {correct}")
print(f"이름(name): {correct.name}")
print(f"값(value): {correct.value}")


## 5. AgenticRAG 클래스 — 모든 부품을 하나로 연결하기

이제 앞서 만든 `MockLLM`, `MockVectorStore`, `SearchStrategy`를 사용해서,
0장에서 그렸던 파이프라인을 실제 클래스로 구현합니다.

`AgenticRAG` 클래스는 아래 메서드들로 구성됩니다.

| 메서드 | 역할 | 0장 그림의 어디에 해당하나요? |
|---|---|---|
| `should_retrieve(query)` | 검색이 필요한지 판단 (True/False) | [1단계] |
| `select_strategy(query)` | 검색 전략을 선택 (SearchStrategy) | [2단계] |
| `retrieve(query, strategy)` | 선택된 전략으로 실제 검색 실행 | [3단계] |
| `query(question)` | 위 모든 단계를 순서대로 실행하는 전체 파이프라인 | 전체 |

그 외에 파이프라인을 돕는 내부용(private) 헬퍼 메서드도 몇 개 추가했습니다.
(이름 앞에 `_`가 붙은 메서드는 "클래스 내부에서만 쓰는 도구"라는 관례적인 표시입니다.)

- `_generate_multi_queries`: MULTI_QUERY 전략에서, 복합 질문을 하위 질문들로 쪼갭니다.
  (원본 코드에서는 이 메서드가 호출되기만 하고 정의되어 있지 않아 실행 시 에러가 났던 부분입니다. 이 버전에서 채워 넣었습니다.)
- `_deduplicate`: 여러 검색 방법을 함께 쓰는 HYBRID/MULTI_QUERY 전략에서, 같은 문서가 중복으로 검색될 수 있으므로 중복을 제거합니다.
- `_format_context`: 검색된 문서 리스트를 LLM이 읽기 좋은 문자열로 정리합니다.
- `_log`: `verbose=True`일 때 진행 상황을 출력합니다.

> **참고**: `__init__`에는 `max_iterations`라는 값도 저장해두는데, 이 "간략화(simplified)" 버전에서는
> 실제로 사용하지 않습니다. 이 값은 "검색 결과가 불충분하면 최대 N번까지 다른 방식으로 재검색을 시도한다"는
> 심화 버전(반복적/iterative Agentic RAG)을 위해 남겨둔 자리입니다. 지금은 존재만 알아두어도 충분합니다.


In [ ]:
class AgenticRAG:
    """
    LLM과 벡터스토어를 받아서, 질문마다 '검색이 필요한지', '어떻게 검색할지'를
    스스로 판단하며 답변을 생성하는 Agentic RAG 파이프라인입니다.
    """

    def __init__(self, llm, vectorstore, max_iterations: int = 3, verbose: bool = True):
        """
        Args:
            llm: 프롬프트(문자열)를 받아 응답(문자열)을 반환하는, 호출 가능한 객체.
                 예) MockLLM 인스턴스, 혹은 실전에서는 실제 Claude/GPT 클라이언트를 감싼 래퍼(wrapper)
            vectorstore: similarity_search / keyword_search 메서드를 가진 객체.
                         예) MockVectorStore 인스턴스, 혹은 실전에서는 Chroma/Pinecone 클라이언트를 감싼 래퍼
            max_iterations: (심화용, 이 버전에서는 미사용) 재검색을 시도할 최대 횟수
            verbose: True로 두면 각 단계의 진행 상황을 출력합니다.
        """
        self.llm = llm
        self.vectorstore = vectorstore
        self.max_iterations = max_iterations
        self.verbose = verbose

    def _log(self, message: str):
        """verbose=True일 때만 진행 상황 메시지를 출력하는 보조 함수입니다."""
        if self.verbose:
            print(message)

    def should_retrieve(self, query: str) -> bool:
        """
        [1단계] 이 질문에 답하기 위해 외부 문서 검색이 필요한지 판단합니다.
        예) "안녕?" 같은 인사말은 검색이 필요 없다고(False) 판단되어야 합니다.
        """
        self._log("\n[1단계] 검색 필요 여부 판단 중...")

        prompt = f"Does this question require external knowledge? '{query}' Answer yes/no."
        response = self.llm(prompt)

        result = "yes" in response.lower()
        self._log(f"   -> 검색 필요 여부: {result}")
        return result

    def select_strategy(self, query: str) -> SearchStrategy:
        """
        [2단계] 검색이 필요하다고 판단되면, 세 가지 전략 중 하나를 선택합니다.
        """
        self._log("[2단계] 검색 전략 선택 중...")

        prompt = f"""Select search strategy for: '{query}'
        Options: semantic, multi_query, hybrid
        Consider: factual=semantic, complex=multi_query, mixed=hybrid"""
        response = self.llm(prompt)

        try:
            strategy = SearchStrategy(response.strip().lower())
        except ValueError:
            # 실제 LLM은 가끔 "semantic" 대신 "Semantic search would work well" 처럼
            # 예상과 다른 형식으로 답할 수 있습니다. 이런 경우를 대비해
            # 프로그램이 멈추지 않도록 기본 전략(semantic)으로 안전하게 대체합니다.
            self._log(f"   ! 예상치 못한 응답 '{response}' -> 기본 전략(semantic)으로 대체")
            strategy = SearchStrategy.SEMANTIC

        self._log(f"   -> 선택된 전략: {strategy.value}")
        return strategy

    def _generate_multi_queries(self, query: str) -> List[str]:
        """(MULTI_QUERY 전략에서 사용) 복합 질문을 여러 개의 하위 질문으로 쪼갭니다."""
        prompt = f"Break down this question into 2-3 simpler sub-questions: '{query}'"
        response = self.llm(prompt)

        queries = [q.strip() for q in response.split("\n") if q.strip()]
        self._log(f"   -> 하위 질문 {len(queries)}개로 분리: {queries}")
        return queries

    def _deduplicate(self, docs: List[Dict]) -> List[Dict]:
        """content가 동일한 중복 문서를 제거합니다. (HYBRID/MULTI_QUERY에서 같은 문서가 겹쳐 검색될 수 있음)"""
        seen = set()
        unique_docs = []
        for doc in docs:
            if doc["content"] not in seen:
                seen.add(doc["content"])
                unique_docs.append(doc)
        return unique_docs

    def retrieve(self, query: str, strategy: SearchStrategy) -> List[Dict]:
        """[3단계] 선택된 전략에 따라 실제 검색을 수행합니다."""
        self._log(f"[3단계] '{strategy.value}' 전략으로 검색 실행 중...")

        if strategy == SearchStrategy.SEMANTIC:
            # 질문을 그대로 1회, 의미 기반 검색
            results = self.vectorstore.similarity_search(query, k=3)

        elif strategy == SearchStrategy.MULTI_QUERY:
            # 질문을 하위 질문 여러 개로 쪼갠 뒤, 각각 검색해서 결과를 합침
            sub_queries = self._generate_multi_queries(query)
            results = []
            for sub_query in sub_queries:
                results.extend(self.vectorstore.similarity_search(sub_query, k=2))
            results = self._deduplicate(results)

        elif strategy == SearchStrategy.HYBRID:
            # 의미 기반 검색 + 키워드 검색을 함께 실행하고 결과를 합침
            semantic_results = self.vectorstore.similarity_search(query, k=2)
            keyword_results = self.vectorstore.keyword_search(query, k=2)
            results = self._deduplicate(semantic_results + keyword_results)

        else:
            # 여기 도달할 일은 없지만, 예상치 못한 전략값에 대비한 안전장치입니다.
            results = []

        self._log(f"   -> 검색된 문서 수: {len(results)}개")
        return results

    def _format_context(self, retrieved_docs: List[Dict]) -> str:
        """
        검색된 문서 리스트를 LLM이 읽기 좋은 하나의 문자열로 정리합니다.

        (참고) 이 정리 과정 없이 딕셔너리 리스트를 그대로 프롬프트에 넣으면
        "[{'content': '...', 'score': 2}, ...]" 처럼 지저분한 문자열이 되어버립니다.
        실전에서는 이렇게 LLM이 읽기 좋은 형태로 context를 다듬는 과정이 중요합니다.
        """
        if not retrieved_docs:
            return "(검색된 문서 없음)"
        return "\n".join(
            f"[문서 {i}] {doc['content']}"
            for i, doc in enumerate(retrieved_docs, start=1)
        )

    def query(self, question: str) -> str:
        """
        전체 Agentic RAG 파이프라인을 실행합니다.

        흐름:
          질문 입력
            -> [1단계] 검색이 필요한가?
                -> 필요 없으면: 바로 LLM이 답변하고 종료
                -> 필요하면: 아래 단계 계속 진행
            -> [2단계] 어떤 전략으로 검색할까?
            -> [3단계] 선택한 전략으로 검색 실행
            -> [4단계] 검색 결과(context) + 질문을 LLM에게 주고 최종 답변 생성
        """
        self._log(f"\n{'=' * 60}\n질문: {question}\n{'=' * 60}")

        # [1단계] 검색 필요 여부 판단
        if not self.should_retrieve(question):
            self._log("   -> 검색 없이 바로 답변합니다.")
            return self.llm(question)

        # [2단계] 검색 전략 선택
        strategy = self.select_strategy(question)

        # [3단계] 검색 실행
        retrieved_docs = self.retrieve(question, strategy)

        # [4단계] 검색 결과를 바탕으로 최종 답변 생성
        self._log("[4단계] 검색 결과를 바탕으로 답변 생성 중...")
        context = self._format_context(retrieved_docs)
        prompt = f"""Based on context, answer the question.
        Context: {context}
        Question: {question}
        Answer:"""
        answer = self.llm(prompt)

        return answer


print("AgenticRAG 클래스 정의 완료.")


## 6. 인스턴스 생성하기

앞서 만들어둔 `vectorstore` 객체를 그대로 재사용하고, `llm`은 이번 절에서 verbose=True로 새로 만들어
`AgenticRAG`에 전달합니다. 이렇게 "이미 만들어둔 부품을 그대로 조립"할 수 있는 것은, `AgenticRAG`가
특정 LLM/벡터스토어 구현에 의존하지 않고 "호출하면 응답을 주는 llm"과 "검색 메서드를 가진 vectorstore"라는
**역할(인터페이스)**에만 의존하도록 설계했기 때문입니다. (나중에 이 자리에 진짜 Claude API, 진짜 Chroma DB를
끼워 넣어도 `AgenticRAG` 클래스 코드는 바꿀 필요가 없습니다.)


In [ ]:
# 이번에는 verbose=True로 새로 만들어서, LLM 호출 내역까지 전부 확인할 수 있게 합니다.
llm = MockLLM(verbose=True)
agent = AgenticRAG(llm=llm, vectorstore=vectorstore, verbose=True)

print("AgenticRAG 인스턴스가 준비되었습니다.")
print(f"연결된 vectorstore가 가진 문서 수: {len(vectorstore.documents)}개")


## 7. 부품별로 테스트해보기

전체 파이프라인(`query`)을 한 번에 실행하기 전에, 각 단계별 메서드를 하나씩 따로 호출해보며
입력과 출력을 확인해봅시다. 이렇게 하면 나중에 전체 파이프라인을 실행했을 때
"지금 내부에서 무슨 일이 일어나고 있는지" 훨씬 쉽게 이해할 수 있습니다.

### 7-1. should_retrieve — 검색이 필요한가?


In [ ]:
print("### 인사말 ###")
result1 = agent.should_retrieve("안녕하세요! 오늘 기분이 어때요?")
print(f"결과: {result1}\n")

print("### 사실을 묻는 질문 ###")
result2 = agent.should_retrieve("연차는 며칠인가요?")
print(f"결과: {result2}")


### 7-2. select_strategy — 어떤 방식으로 검색할까?

세 가지 질문을 넣어서, 질문의 생김새에 따라 정말로 다른 전략이 선택되는지 확인해봅시다.


In [ ]:
questions_for_strategy = [
    "연차는 1년에 며칠 주어지나요?",                                        # 단순한 사실 질문
    "재택근무는 어떻게 신청하나요? 그리고 출장 숙박비 한도는 얼마인가요?",   # 복합 질문
    "비밀번호를 잊어버렸을 때 IT 헬프데스크 내선번호가 몇 번인가요?",         # 개념 + 정확한 정보
]

for q in questions_for_strategy:
    strategy = agent.select_strategy(q)
    print(f"질문: {q}\n-> 선택된 전략: {strategy.value}\n")


### 7-3. retrieve — 실제로 검색을 실행하면?

`select_strategy`가 골라준 전략을 그대로 `retrieve`에 넘겨서, 실제로 어떤 문서가 검색되는지 확인해봅시다.


In [ ]:
print("### SEMANTIC 전략으로 검색 ###")
docs1 = agent.retrieve("연차는 1년에 며칠 주어지나요?", SearchStrategy.SEMANTIC)
print(json.dumps(docs1, ensure_ascii=False, indent=2))

print("\n### HYBRID 전략으로 검색 ###")
docs2 = agent.retrieve("비밀번호를 잊어버렸을 때 IT 헬프데스크 내선번호가 몇 번인가요?", SearchStrategy.HYBRID)
print(json.dumps(docs2, ensure_ascii=False, indent=2))


## 8. 전체 파이프라인 실행해보기 (`query` 메서드)

이제 부품이 아니라, `query()` 하나로 전체 파이프라인을 실행해봅시다.
아래 4가지 질문은 각각 "인사말 / 단순 사실 / 복합 질문 / 개념+정확한 정보"라는
서로 다른 성격을 가지고 있어서, 파이프라인이 각기 다른 경로로 흘러가는 것을 확인할 수 있습니다.


### 예시 1: 검색이 필요 없는 질문 (인사말)

`should_retrieve`가 False를 반환해서, 검색 없이 바로 답변할 것으로 예상됩니다.


In [ ]:
answer1 = agent.query("안녕하세요! 오늘 기분이 어때요?")
print(f"\n\n★ 최종 답변: {answer1}")


### 예시 2: 단순 사실 질문 -> SEMANTIC 전략

하나의 명확한 사실(연차 일수)을 묻고 있으므로 SEMANTIC 전략이 선택될 것으로 예상됩니다.


In [ ]:
answer2 = agent.query("연차는 1년에 며칠 주어지나요?")
print(f"\n\n★ 최종 답변: {answer2}")


### 예시 3: 복합 질문 -> MULTI_QUERY 전략

"그리고"로 연결된 두 개의 하위 질문이 섞여 있으므로 MULTI_QUERY 전략이 선택되고,
질문이 두 개로 쪼개져서 각각 검색될 것으로 예상됩니다.


In [ ]:
answer3 = agent.query("재택근무는 어떻게 신청하나요? 그리고 출장 숙박비 한도는 얼마인가요?")
print(f"\n\n★ 최종 답변: {answer3}")


### 예시 4: 개념 + 정확한 정보 질문 -> HYBRID 전략

"비밀번호 초기화"라는 개념과 "내선번호"라는 정확한 정보를 동시에 묻고 있으므로
HYBRID 전략(의미 기반 + 키워드)이 선택될 것으로 예상됩니다.


In [ ]:
answer4 = agent.query("비밀번호를 잊어버렸을 때 IT 헬프데스크 내선번호가 몇 번인가요?")
print(f"\n\n★ 최종 답변: {answer4}")


## 9. 결과 한눈에 비교하기

verbose 출력 없이, 4개의 질문이 각각 어떤 경로를 탔는지 깔끔하게 정리해봅시다.


In [ ]:
# 이번에는 verbose=False인 조용한 agent를 새로 만들어서, 결과만 깔끔하게 뽑아봅니다.
quiet_llm = MockLLM(verbose=False)
quiet_agent = AgenticRAG(llm=quiet_llm, vectorstore=vectorstore, verbose=False)

summary_questions = [
    "안녕하세요! 오늘 기분이 어때요?",
    "연차는 1년에 며칠 주어지나요?",
    "재택근무는 어떻게 신청하나요? 그리고 출장 숙박비 한도는 얼마인가요?",
    "비밀번호를 잊어버렸을 때 IT 헬프데스크 내선번호가 몇 번인가요?",
]

print("=" * 70)
for q in summary_questions:
    needs = quiet_agent.should_retrieve(q)
    print(f"질문: {q}")
    if needs:
        strategy = quiet_agent.select_strategy(q)
        docs = quiet_agent.retrieve(q, strategy)
        print(f"  - 검색 필요: 예 (선택된 전략: {strategy.value}, 검색된 문서: {len(docs)}개)")
    else:
        print(f"  - 검색 필요: 아니오 (검색 없이 바로 답변)")
    print("-" * 70)


## 10. 직접 해보기

이제 직접 질문을 바꿔가며 실험해보세요! 아래를 시도해볼 수 있습니다.

- `my_question`을 자유롭게 바꿔서 어떤 전략이 선택되는지 관찰해보기
- `MockVectorStore`의 `documents` 리스트에 새 문서를 추가하고, 그 문서를 검색해보기
- `MockLLM._decide_strategy`의 `multi_signals`/`exact_signals` 키워드를 바꿔보고 결과가 어떻게 달라지는지 확인해보기


In [ ]:
# 아래 질문을 원하는 내용으로 바꿔서 실행해보세요.
my_question = "회의실은 얼마나 미리 예약해야 하나요?"

my_answer = agent.query(my_question)
print(f"\n\n★ 최종 답변: {my_answer}")


## 11. 정리 및 다음 단계

### 이 노트북에서 배운 것

1. **Agentic RAG의 핵심**은 "검색 여부"와 "검색 방법"을 고정하지 않고, LLM이 질문마다 스스로 판단하게 만드는 것입니다.
2. 하나의 LLM이라도 **프롬프트(지시문)를 다르게 설계**하면 판단자·검색 전략가·답변자 등 여러 역할을 수행할 수 있습니다.
3. `MockLLM`, `MockVectorStore`처럼 실제 의존성을 흉내 낸 가짜 객체로도 파이프라인의 **구조**를 완전히 익힐 수 있습니다.
4. `AgenticRAG` 클래스는 "호출 가능한 llm"과 "검색 메서드를 가진 vectorstore"라는 역할에만 의존하도록 설계되어 있어서,
   Mock을 실제 API로 교체해도 `AgenticRAG` 클래스 자체는 손댈 필요가 없습니다.

### 원본 코드에서 고친 부분 (참고용)

- `MULTI_QUERY` 등의 잘못된 들여쓰기로 인한 문법 오류 수정
- 호출은 되지만 정의되어 있지 않던 `_generate_multi_queries` 메서드 구현
- 실행 가능한 `MockLLM`/`MockVectorStore`를 추가해서, API 키 없이 직접 실행 가능하도록 구성
- 검색 결과(딕셔너리 리스트)를 사람이 읽기 좋은 문자열로 정리하는 `_format_context` 추가
- HYBRID/MULTI_QUERY 전략에서 중복 문서를 제거하는 `_deduplicate` 추가
- `select_strategy`에서 LLM이 예상 밖의 값을 반환했을 때를 대비한 예외 처리 추가
- 각 단계의 진행 상황을 확인할 수 있는 `verbose`/`_log` 로그 기능 추가

### 실제 서비스에 적용하려면?

`MockLLM`과 `MockVectorStore`를 실제 구현으로 교체하면 됩니다. `AgenticRAG` 클래스는 그대로 재사용할 수 있습니다.

```python
# (아래는 실행용 코드가 아니라, 실전에서 어떻게 바뀌는지 보여주는 참고용 예시입니다)

import anthropic

class ClaudeLLM:
    def __init__(self, api_key: str, model: str = "claude-sonnet-5"):
        # 최신 모델명은 https://docs.claude.com 에서 확인하세요.
        self.client = anthropic.Anthropic(api_key=api_key)
        self.model = model

    def __call__(self, prompt: str) -> str:
        response = self.client.messages.create(
            model=self.model,
            max_tokens=1000,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.content[0].text

# 실제 벡터스토어(예: Chroma)도 similarity_search / keyword_search 메서드를 갖춘
# 래퍼(wrapper) 클래스로 감싸서 준비하면, AgenticRAG에 그대로 꽂아 쓸 수 있습니다.

# real_agent = AgenticRAG(llm=ClaudeLLM(api_key="..."), vectorstore=real_vectorstore)
```

### 더 나아가기 (심화 학습 아이디어)

- `max_iterations`를 실제로 활용해서, "검색 결과가 충분한지" LLM이 판단하고 부족하면 다른 전략으로 재검색하는
  **반복적(iterative) Agentic RAG**를 구현해보세요.
- `_decide_strategy`의 규칙 기반 로직을 실제 LLM 호출로 교체하면, 더 다양한 문장 패턴에도 유연하게 대응할 수 있습니다.
- `MockVectorStore`의 키워드 매칭을 `sentence-transformers` 같은 실제 임베딩 모델로 교체해서, 진짜 의미 기반 검색을 경험해보세요.
